# Genetic Programming Notebook.
In this notebook, I follow along Burakk Anber's tutorial "[Machine Learning: Introduction to Genetic Algorithms]((https://burakkanber.com/blog/machine-learning-genetic-algorithms-part-1-javascript/)". In it, Burakk Anber develops a genetic algorithm for steering random data toward a predefined string. This is useful for cool effects, and serves as an introduction to machine learning. 

All I do is update the code to contemporary javascript/typescript, as Anber's blog post featuring the code was written 14 years ago.

In [2]:
Gene = class Gene {
  code: String = '';
  cost: Number = 9999;
  
  constructor(code: String){
    if(code) this.code = code;
  }

  random(length: Number): void {
    while(length--){
      this.code += String.fromCharCode(Math.floor(Math.random()*255));
    }
  }

  mutate(chance): void {
    if(Math.random() > chance) return;

    const index = Math.floor(Math.random() * this.code.length);
    const upOrDown = Math.random() <= 0.5 ? -1 : 1;
    const newChar = String.fromCharCode(this.code.charCodeAt(index) + upOrDown);
    let newString = '';
    for(let i=0; i < this.code.length; i++){
      if(i === index) newString += newChar;
      else newString += this.code[i];
    }

    this.code = newString;
  }

  mate(gene: Gene): Array<Gene> {
    const pivot = Math.round(this.code.length / 2) - 1;

    const child1 = this.code.substr(0, pivot) + gene.code.substr(pivot);
    const child2 = gene.code.substr(0, pivot) + this.code.substr(pivot);

    return [new Gene(child1), new Gene(child2)];
  }

  calcCost(compareTo: String): Number {
    let total = 0;
    for(let i=0; i < this.code.length; i++){
      const diff = this.code.charCodeAt(i) - compareTo.charCodeAt(i);
      total += diff * diff;
    }

    this.cost = total;
  }
}

In [2]:
Population = class Population {
  members: Array<Gene> = [];
  goal: String;
  generationNumber = 0;
  
  constructor(goal: String, size: Number){
    this.goal = goal;
    while(size--){
      let gene = new Gene();
      gene.random(this.goal.length);
      this.members.push(gene);
    }
  }

  sort(){
    this.members.sort((a, b) => a.cost - b.cost);
  }

  generation(): void {
    this.members.forEach(member => member.calcCost(this.goal));

    this.sort();
    this.display();

    const children = this.members[0].mate(this.members[1]);
    this.members.splice(this.members.length - 2, 2, ...children);

    for(let i=0; i < this.members.length; i++){
      const member = this.members[i];
      
      member.mutate(0.5);
      member.calcCost(this.goal);
      if(member.code === this.goal){
        this.sort();
        this.display();
        return true;
      }
    }

    this.generationNumber++;
    setTimeout(() => this.generation(), 20);
  }

  display(){
    output.innerHTML = `
      <h2>Generation ${this.generationNumber}</h2>
      <ul>
        <li>${this.members.map((member) => member.code + ` (${member.cost})`).join(`</li><li>`)}</li>
      </ul>
    `
  }
}


      <h2>Generation 1013</h2>
      <ul>
        <li>Hello, World! (0)</li><li>Hello, Wpqld! (2)</li><li>Hdllo, Worle! (2)</li><li>Helmo, Woqld! (2)</li><li>Hello, Vorle! (2)</li><li>Iello, Worle! (2)</li><li>Hellp, Wnrld! (2)</li><li>Helko, Wprld! (2)</li><li>Helmo, Wprle! (3)</li><li>Hemmo, Woqld! (3)</li><li>Hello- Worle  (3)</li><li>Gdllo, Worle! (3)</li><li>Helko, Wprle" (4)</li><li>Hfllp, Worme! (4)</li><li>Gdllo,Worle! (4)</li><li>Geklo, Worle" (4)</li><li>Gelko, Worme" (5)</li><li>Hemko- Wosle! (5)</li><li>Hdlln, Wnrke" (6)</li><li>Hello, Woqld! (9999)</li>
      </ul>
    

In [2]:
const population = new Population("Hello, World!", 20);
population.generation()